In [1]:
# Put this in a cell BEFORE importing paddleocr / creating PaddleOCR
import os
os.environ["FLAGS_use_mkldnn"] = "0"     # disable oneDNN (MKLDNN)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # sometimes helps on Windows envs
os.environ["FLAGS_use_new_executor"] = "0"

In [2]:
# Cell 2 — Imports + paths
import os, json
from pathlib import Path
from typing import List, Dict, Any, Tuple

import cv2
from PIL import Image, ImageDraw

INPUT_PNG_DIR = Path("./data/jap")          # <-- change if needed
OUT_JSON_DIR  = Path("./outputs/ocr_json_jap_new")
OUT_VIS_DIR   = Path("./outputs/ocr_vis_jap_new")

OUT_JSON_DIR.mkdir(parents=True, exist_ok=True)
OUT_VIS_DIR.mkdir(parents=True, exist_ok=True)

png_files = sorted(INPUT_PNG_DIR.glob("*.png"))
print("Found PNGs:", len(png_files))
if png_files:
    print("Example:", png_files[0])

Found PNGs: 6
Example: data\jap\pg_1.png


In [3]:
import cv2

def resize_keep_aspect(img_bgr, max_side=2400):
    h, w = img_bgr.shape[:2]
    scale = min(max_side / max(h, w), 1.0)  # only shrink if needed
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    resized = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
    return resized, scale

def preprocess_for_manga(img_bgr, scale=1.0, max_side=2400, invert=False):
    if scale != 1.0:
        img_bgr = cv2.resize(
            img_bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC
        )

    img_bgr, resize_scale = resize_keep_aspect(img_bgr, max_side=max_side)

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.fastNlMeansDenoising(gray, h=8)

    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    if invert:
        th = 255 - th

    th_bgr = cv2.cvtColor(th, cv2.COLOR_GRAY2BGR)
    return th_bgr, resize_scale

# Cell 4 — Helpers: bbox conversion + JSON save
def quad_to_xyxy(quad: List[List[float]]) -> Tuple[int, int, int, int]:
    """
    quad: [[x1,y1],[x2,y2],[x3,y3],[x4,y4]] (PaddleOCR det polygon)
    returns (x_min, y_min, x_max, y_max)
    """
    xs = [p[0] for p in quad]
    ys = [p[1] for p in quad]
    return int(round(min(xs))), int(round(min(ys))), int(round(max(xs))), int(round(max(ys)))

def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def safe_str(x) -> str:
    return "" if x is None else str(x)

from typing import Any, Dict
from pathlib import Path
from PIL import Image
import numpy as np

def safe_str(x):
    return "" if x is None else str(x)

def quad_to_xyxy(quad):
    xs = [float(p[0]) for p in quad]
    ys = [float(p[1]) for p in quad]
    return [min(xs), min(ys), max(xs), max(ys)]

import re

def keep_item(text, conf, bbox_xyxy):
    if conf is not None and conf < 0.5:
        return False

    text = text.strip()
    if not text:
        return False

    # keep mostly English-looking text
    if not re.search(r"[A-Za-z]", text):
        return False

    x1, y1, x2, y2 = bbox_xyxy
    w, h = x2 - x1, y2 - y1

    # remove tiny junk
    if w < 8 or h < 8:
        return False

    return True

def normalize_paddle_result(raw_ocr_output, image_shape, image_path=None):
    H, W = image_shape[:2]

    if isinstance(raw_ocr_output, list):
        data = raw_ocr_output[0]
    else:
        data = raw_ocr_output

    polys = list(data.get("dt_polys", []))
    texts = list(data.get("rec_texts", []))
    scores = list(data.get("rec_scores", []))

    n = min(len(polys), len(texts), len(scores))
    items = []

    for idx in range(n):
        quad = polys[idx]
        if isinstance(quad, np.ndarray):
            quad = quad.tolist()

        quad = [[float(p[0]), float(p[1])] for p in quad]
        xs = [p[0] for p in quad]
        ys = [p[1] for p in quad]

        x1 = int(round(max(0, min(W, min(xs)))))
        y1 = int(round(max(0, min(H, min(ys)))))
        x2 = int(round(max(0, min(W, max(xs)))))
        y2 = int(round(max(0, min(H, max(ys)))))

        items.append({
            "id": idx,
            "bbox_xyxy": [x1, y1, x2, y2],
            "poly": quad,
            "text": str(texts[idx]) if texts[idx] is not None else "",
            "conf": float(scores[idx]) if scores[idx] is not None else None,
        })

    return {
        "image": str(image_path) if image_path else None,
        "width": W,
        "height": H,
        "items": items,
        "engine": "paddleocr",
        "lang": "English",
    }

In [4]:
# Cell 3 — Initialize PaddleOCR (Japanese)
from paddleocr import PaddleOCR

# Common settings:
# - lang="japan" gives Japanese model
# - use_angle_cls=True helps with rotated text
# - det_db_box_thresh, det_db_thresh can be tuned later; keep defaults for milestone
def initialize_paddle_ocr(db_thresh=0.3, box_thresh=0.6, unclip_ratio=1.5, limit_side_len=4096):
    ocr = PaddleOCR(
        lang="japan",
        use_angle_cls=True,
        det_db_thresh=db_thresh,
        det_db_box_thresh=box_thresh,
        det_db_unclip_ratio=unclip_ratio,
        # keep bigger inputs
        det_limit_side_len=limit_side_len,
    )
    print("✅ PaddleOCR initialized")
    return ocr

C:\Users\abcdj\AppData\Roaming\Python\Python310\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.0.post2)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [5]:
import itertools
from pathlib import Path
from PIL import Image
import cv2
import numpy as np

# ---------- helpers ----------
def run_one_setting(
    test_img_path,
    db_thresh,
    box_thresh,
    unclip_ratio,
    limit_side_len=2048,
    save_dir="ocr_grid_outputs_og",
):
    ocr = initialize_paddle_ocr(
        db_thresh=db_thresh,
        box_thresh=box_thresh,
        unclip_ratio=unclip_ratio,
        limit_side_len=limit_side_len,
    )

    img_bgr = cv2.imread(str(test_img_path))
    proc_bgr, meta = preprocess_for_manga(img_bgr)

    raw = ocr.predict(proc_bgr)
    norm = normalize_paddle_result(raw, proc_bgr.shape, image_path=test_img_path)

    norm["items"] = [
        it for it in norm["items"]
        if keep_item(it["text"], it["conf"], it["bbox_xyxy"])
    ]

    vis = proc_bgr.copy()
    for it in norm["items"]:
        pts = np.array(it["poly"], dtype=np.int32)
        cv2.polylines(vis, [pts], True, (0, 0, 255), 2)

    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    safe_db = str(db_thresh).replace(".", "p")
    safe_box = str(box_thresh).replace(".", "p")
    safe_unclip = str(unclip_ratio).replace(".", "p")

    out_name = f"db{safe_db}_box{safe_box}_unclip{safe_unclip}_limit{limit_side_len}.png"
    out_path = save_dir / out_name
    Image.fromarray(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)).save(out_path)

    return {
        "db_thresh": db_thresh,
        "box_thresh": box_thresh,
        "unclip_ratio": unclip_ratio,
        "limit_side_len": limit_side_len,
        "num_items": len(norm["items"]),
        "output_path": str(out_path),
        "items": norm["items"],
    }


def try_paddle_combinations(
    test_img_path,
    db_thresh_values=(0.3, 0.5, 0.7, 0.8),
    box_thresh_values=(0.5, 0.6, 0.7),
    unclip_ratio_values=(1.2, 1.5, 1.8),
    limit_side_len=(2048),
    save_dir="ocr_grid_outputs",
):
    results = []

    combos = list(itertools.product(
        db_thresh_values,
        box_thresh_values,
        unclip_ratio_values
    ))

    print(f"Running {len(combos)} combinations...")

    for db_thresh, box_thresh, unclip_ratio in combos:
        # Generate the output filename FIRST
        safe_db = str(db_thresh).replace(".", "p")
        safe_box = str(box_thresh).replace(".", "p")
        safe_unclip = str(unclip_ratio).replace(".", "p")
        out_name = f"db{safe_db}_box{safe_box}_unclip{safe_unclip}_limit{limit_side_len}.png"
        out_path = Path(save_dir) / out_name
        
        # SKIP if file already exists
        if out_path.exists():
            print(f"  ✓ {out_name} already exists, skipping...")
            continue
        
        print(
            f"Trying db={db_thresh}, box={box_thresh}, "
            f"unclip={unclip_ratio}, limit={limit_side_len}"
        )
        print(
            f"Trying db={db_thresh}, box={box_thresh}, "
            f"unclip={unclip_ratio}, limit={limit_side_len}"
        )
        try:
            res = run_one_setting(
                test_img_path=test_img_path,
                db_thresh=db_thresh,
                box_thresh=box_thresh,
                unclip_ratio=unclip_ratio,
                limit_side_len=limit_side_len,
                save_dir=save_dir,
            )
            results.append(res)
            print(f"  -> kept {res['num_items']} boxes")
        except Exception as e:
            raise Exception(e)

    # sort by number of kept boxes, descending
    results = sorted(results, key=lambda x: x["num_items"], reverse=True)
    return results

In [6]:
test_img_path = png_files[1]
test_img_path

WindowsPath('data/jap/pg_2.png')

In [7]:
results = try_paddle_combinations(
    test_img_path,
    db_thresh_values=(0.2, 0.4, 0.6, 0.8),
    box_thresh_values=(0.2, 0.35, 0.5, 0.65, 0.8),
    unclip_ratio_values=(1.2, 1.5, 1.8),
    limit_side_len=(2048),
    save_dir="ocr_grid_outputs_og"
)

results[:5]

Running 60 combinations...
  ✓ db0p2_box0p2_unclip1p2_limit2048.png already exists, skipping...
  ✓ db0p2_box0p2_unclip1p5_limit2048.png already exists, skipping...
  ✓ db0p2_box0p2_unclip1p8_limit2048.png already exists, skipping...
  ✓ db0p2_box0p35_unclip1p2_limit2048.png already exists, skipping...
  ✓ db0p2_box0p35_unclip1p5_limit2048.png already exists, skipping...
  ✓ db0p2_box0p35_unclip1p8_limit2048.png already exists, skipping...
  ✓ db0p2_box0p5_unclip1p2_limit2048.png already exists, skipping...
  ✓ db0p2_box0p5_unclip1p5_limit2048.png already exists, skipping...
  ✓ db0p2_box0p5_unclip1p8_limit2048.png already exists, skipping...
  ✓ db0p2_box0p65_unclip1p2_limit2048.png already exists, skipping...
  ✓ db0p2_box0p65_unclip1p5_limit2048.png already exists, skipping...
  ✓ db0p2_box0p65_unclip1p8_limit2048.png already exists, skipping...
  ✓ db0p2_box0p8_unclip1p2_limit2048.png already exists, skipping...
  ✓ db0p2_box0p8_unclip1p5_limit2048.png already exists, skipping...
 

C:\Users\abcdj\AppData\Local\Temp\ipykernel_24636\1376805615.py:9: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(
C:\Users\abcdj\AppData\Local\Temp\ipykernel_24636\1376805615.py:9: DeprecationWarning: The parameter `det_db_thresh` has been deprecated and will be removed in the future. Please use `text_det_thresh` instead.
  ocr = PaddleOCR(
C:\Users\abcdj\AppData\Local\Temp\ipykernel_24636\1376805615.py:9: DeprecationWarning: The parameter `det_db_box_thresh` has been deprecated and will be removed in the future. Please use `text_det_box_thresh` instead.
  ocr = PaddleOCR(
C:\Users\abcdj\AppData\Local\Temp\ipykernel_24636\1376805615.py:9: DeprecationWarning: The parameter `det_db_unclip_ratio` has been deprecated and will be removed in the future. Please use `text_det_unclip_ratio` instead.
  ocr = PaddleOCR(
C:\Users\abcdj\AppData\Local\Temp\ipykernel_24636\13768

✅ PaddleOCR initialized


Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\abcdj\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.


  -> kept 6 boxes
Trying db=0.4, box=0.5, unclip=1.2, limit=2048
Trying db=0.4, box=0.5, unclip=1.2, limit=2048


...

Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\abcdj\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.


  -> kept 2 boxes
Trying db=0.8, box=0.8, unclip=1.8, limit=2048
Trying db=0.8, box=0.8, unclip=1.8, limit=2048


Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\abcdj\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\abcdj\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\abcdj\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\abcdj\.paddlex\official_models\PP-OCRv5_server_rec`.


✅ PaddleOCR initialized
  -> kept 3 boxes


[{'db_thresh': 0.4,
  'box_thresh': 0.35,
  'unclip_ratio': 1.8,
  'limit_side_len': 2048,
  'num_items': 6,
  'output_path': 'ocr_grid_outputs_og\\db0p4_box0p35_unclip1p8_limit2048.png',
  'items': [{'id': 0,
    'bbox_xyxy': [32, 3, 127, 251],
    'poly': [[34.0, 3.0], [127.0, 4.0], [124.0, 251.0], [32.0, 250.0]],
    'text': 'ueN',
    'conf': 0.5624783635139465},
   {'id': 10,
    'bbox_xyxy': [418, 388, 542, 436],
    'poly': [[419.0, 388.0], [542.0, 391.0], [541.0, 436.0], [418.0, 433.0]],
    'text': 'ANCHOR',
    'conf': 0.9933257699012756},
   {'id': 13,
    'bbox_xyxy': [788, 609, 823, 624],
    'poly': [[788.0, 609.0], [823.0, 609.0], [823.0, 624.0], [788.0, 624.0]],
    'text': 'h',
    'conf': 0.6165883541107178},
   {'id': 25,
    'bbox_xyxy': [703, 845, 742, 875],
    'poly': [[703.0, 855.0], [735.0, 845.0], [742.0, 864.0], [709.0, 875.0]],
    'text': 'M',
    'conf': 0.5857818126678467},
   {'id': 35,
    'bbox_xyxy': [960, 1038, 971, 1054],
    'poly': [[960.0, 1038.0